
# 1. Imports & APIs
Importing essential PySpark SQL functions for data transformation and aggregation.

In [0]:
from pyspark.sql import functions as F


# 2. Marketing Transactions Refinement
Cleaning the orders table by filtering for 'delivered' status and consolidating fragmented payments into a single order value.

In [0]:
print("Processing Silver Transactions...")

# Loading Bronze tables
df_orders = spark.table("workspace.bronze_marketing_project.bronze_orders")
df_payments = spark.table("workspace.bronze_marketing_project.bronze_order_payments")

# Filtering delivered orders with valid timestamps
df_orders_clean = df_orders.filter(
    (F.col("order_status") == "delivered") & 
    (F.col("order_purchase_timestamp").isNotNull())
)

# Aggregating payments by order_id
df_payments_agg = df_payments.groupBy("order_id").agg(
    F.sum("payment_value").alias("total_order_value")
)

# Joining and selecting core transaction features
silver_transactions = df_orders_clean.join(df_payments_agg, on="order_id", how="inner") \
    .select("order_id", "customer_id", "order_purchase_timestamp", "total_order_value")

# Saving to Silver Schema with Table-Level Documentation
(silver_transactions.write
  .format("delta")
  .mode("overwrite")
  .option("comment", "Cleaned transactions: Only delivered orders with consolidated payment values per order.")
  .saveAsTable("workspace.silver_marketing_project.silver_marketing_transactions")
)
print("✓ Transactions table saved.")


# 3. Data Governance: Transactions Columns
Adding business descriptions to each column in the Unity Catalog for data discovery.

In [0]:
%sql
COMMENT ON COLUMN workspace.silver_marketing_project.silver_marketing_transactions.order_id IS 'Unique identifier for the completed purchase';
COMMENT ON COLUMN workspace.silver_marketing_project.silver_marketing_transactions.customer_id IS 'Transactional customer ID (changes per order)';
COMMENT ON COLUMN workspace.silver_marketing_project.silver_marketing_transactions.order_purchase_timestamp IS 'Exact date and time the order was placed';
COMMENT ON COLUMN workspace.silver_marketing_project.silver_marketing_transactions.total_order_value IS 'Total monetary value paid, aggregating all payment methods (Credit Card, etc.)';


# 4. Customer Profile Consolidation
Deduplicating geographical data to prevent Cartesian explosions and mapping unique customer IDs to their respective regions.

In [0]:
print("Processing Silver Customer Profiles...")

# Loading Customer and Geo data
df_customers = spark.table("workspace.bronze_marketing_project.bronze_customers")
df_geo = spark.table("workspace.bronze_marketing_project.bronze_geolocation")

# Deduplicating geo coordinates by zip code to prevent data explosion
df_geo_clean = df_geo.dropDuplicates(["geolocation_zip_code_prefix"])

# Merging customer IDs with clean geo data
silver_customers = df_customers.join(
    df_geo_clean, 
    df_customers.customer_zip_code_prefix == df_geo_clean.geolocation_zip_code_prefix, 
    how="left"
).select(
    "customer_id", 
    "customer_unique_id", 
    "customer_city", 
    "customer_state", 
    F.col("geolocation_lat").alias("lat"), 
    F.col("geolocation_lng").alias("lng")
)

# Saving to Silver Schema with Table-Level Documentation
(silver_customers.write
  .format("delta")
  .mode("overwrite")
  .option("comment", "Consolidated customer profiles mapped to deduplicated geolocation coordinates.")
  .saveAsTable("workspace.silver_marketing_project.silver_customer_profiles")
)
print("✓ Customer profiles table saved.")


# 5. Data Governance: Customers Columns
Documenting the spatial and identity columns for the analytics team.

In [0]:
%sql
COMMENT ON COLUMN workspace.silver_marketing_project.silver_customer_profiles.customer_id IS 'Transactional ID (Foreign key to orders)';
COMMENT ON COLUMN workspace.silver_marketing_project.silver_customer_profiles.customer_unique_id IS 'True individual identity (The actual human being)';
COMMENT ON COLUMN workspace.silver_marketing_project.silver_customer_profiles.customer_city IS 'Registered city of residence';
COMMENT ON COLUMN workspace.silver_marketing_project.silver_customer_profiles.lat IS 'Geographical Latitude based on Zip Code';
COMMENT ON COLUMN workspace.silver_marketing_project.silver_customer_profiles.lng IS 'Geographical Longitude based on Zip Code';